<a href="https://colab.research.google.com/github/JacobeJonathan/python---pyspark-with-SQL/blob/main/proyectoETL_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance pyspark

## ETL Process: Stock Data (KO, NVDA, AAPL)

This process consists of:
1. **Extraction**: Getting data from `yfinance` using a string list of tickers.
2. **Transformation**: Filtering for 'Date' and 'Close' values.
3. **Loading**: Creating a PySpark DataFrame with the results.

In [2]:
import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
import pandas as pd

# Initialize Spark Session
spark = SparkSession.builder.appName("StockETL").getOrCreate()

# --- 1. EXTRACTION ---
tickers_string = "KO, NVDA, AAPL"
# Convert string to list
tickers_list = [t.strip() for t in tickers_string.split(",")]

raw_data = []
for ticker in tickers_list:
    # Extracting historical data (e.g., last 1 month)
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1mo")

    # --- 2. TRANSFORMATION ---
    # Reset index to get Date as a column and filter needed fields
    hist = hist.reset_index()[['Date', 'Close']]
    hist['Symbol'] = ticker
    raw_data.append(hist)

# Combine all pandas DataFrames
combined_df = pd.concat(raw_data)

# Ensure Date is in the correct format for Spark
combined_df['Date'] = combined_df['Date'].dt.date

# --- 3. LOADING ---
# Define Schema
schema = StructType([
    StructField("fecha", DateType(), True),
    StructField("monto_close", DoubleType(), True),
    StructField("simbolo", StringType(), True)
])

# Create PySpark DataFrame
spark_df = spark.createDataFrame(combined_df, schema=schema)

# Show Result
spark_df.show()

+----------+-----------------+-------+
|     fecha|      monto_close|simbolo|
+----------+-----------------+-------+
|2026-04-08|77.29000091552734|     KO|
|2026-04-09|78.18000030517578|     KO|
|2026-04-10|77.47000122070312|     KO|
|2026-04-13|76.41000366210938|     KO|
|2026-04-14| 75.9000015258789|     KO|
|2026-04-15|75.30999755859375|     KO|
|2026-04-16|75.18000030517578|     KO|
|2026-04-17|75.73999786376953|     KO|
|2026-04-20| 75.4800033569336|     KO|
|2026-04-21|74.69999694824219|     KO|
|2026-04-22|74.62999725341797|     KO|
|2026-04-23|76.27999877929688|     KO|
|2026-04-24|76.62999725341797|     KO|
|2026-04-27|75.44000244140625|     KO|
|2026-04-28| 78.3499984741211|     KO|
|2026-04-29|78.87000274658203|     KO|
|2026-04-30|78.76000213623047|     KO|
|2026-05-01|78.58000183105469|     KO|
|2026-05-04|78.19000244140625|     KO|
|2026-05-05| 78.4800033569336|     KO|
+----------+-----------------+-------+
only showing top 20 rows


In [3]:
# Displaying with a better format using display()
display(spark_df.toPandas().head(10))

,fecha,monto_close,simbolo
0,2026-04-08,77.290001,KO
1,2026-04-09,78.180000,KO
2,2026-04-10,77.470001,KO
3,2026-04-13,76.410004,KO
4,2026-04-14,75.900002,KO
5,2026-04-15,75.309998,KO
6,2026-04-16,75.180000,KO
7,2026-04-17,75.739998,KO
8,2026-04-20,75.480003,KO
9,2026-04-21,74.699997,KO


### Proceso ETL Optimizado
Este proceso extrae datos de `yfinance`, transforma la información para incluir la columna `stock` y carga los resultados en un DataFrame de PySpark.

In [11]:
import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
import pandas as pd

# Iniciar sesión de Spark si no existe
spark = SparkSession.builder.getOrCreate()

# 1. Extracción
simbolos = ["KO", "NVDA", "AAPL"]
datos_lista = []

for s in simbolos:
    ticker = yf.Ticker(s)
    df_hist = ticker.history(period="1mo").reset_index()

    # 2. Transformación
    # Seleccionamos fecha (Date), cierre (Close) y añadimos el nombre 'stock'
    df_temp = df_hist[['Date', 'Close']].copy()
    df_temp['stock'] = s
    df_temp['Date'] = df_temp['Date'].dt.date
    datos_lista.append(df_temp)

# Consolidar datos en Pandas
pandads_final = pd.concat(datos_lista)

# 3. Carga a PySpark
esquema = StructType([
    StructField("fecha", DateType(), True),
    StructField("close", DoubleType(), True),
    StructField("stock", StringType(), True)
])

spark_stocks_df = spark.createDataFrame(pandads_final, schema=esquema)

# Mostrar resultados
spark_stocks_df.show(10)
display(spark_stocks_df.toPandas().head())

+----------+-----------------+-----+
|     fecha|            close|stock|
+----------+-----------------+-----+
|2026-04-08|77.29000091552734|   KO|
|2026-04-09|78.18000030517578|   KO|
|2026-04-10|77.47000122070312|   KO|
|2026-04-13|76.41000366210938|   KO|
|2026-04-14| 75.9000015258789|   KO|
|2026-04-15|75.30999755859375|   KO|
|2026-04-16|75.18000030517578|   KO|
|2026-04-17|75.73999786376953|   KO|
|2026-04-20| 75.4800033569336|   KO|
|2026-04-21|74.69999694824219|   KO|
+----------+-----------------+-----+
only showing top 10 rows


,fecha,close,stock
0,2026-04-08,77.290001,KO
1,2026-04-09,78.180000,KO
2,2026-04-10,77.470001,KO
3,2026-04-13,76.410004,KO
4,2026-04-14,75.900002,KO


### Proceso ETL sin Pandas
Versión optimizada que utiliza listas nativas de Python para cargar los datos en PySpark.

In [12]:
import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Iniciar sesión de Spark
spark = SparkSession.builder.getOrCreate()

# --- PARÁMETROS DE FECHA ---
fecha_inicio_str = "2023-01-01"
fecha_fin_str = "2023-12-31"
simbolos = ["KO", "NVDA", "AAPL"]

# 1. Extracción y Transformación Manual (Sin Pandas)
datos_totales = []

for s in simbolos:
    ticker = yf.Ticker(s)
    hist = ticker.history(start=fecha_inicio_str, end=fecha_fin_str)

    for date, row in hist.iterrows():
        close_val = float(row['Close'])
        datos_totales.append((
            int(date.year),          # Solo el año
            round(close_val, 2),    # precio con 2 dígitos
            s,
            fecha_inicio_str,
            fecha_fin_str
        ))

# 2. Carga a PySpark
esquema = StructType([
    StructField("año", IntegerType(), True),
    StructField("precio", DoubleType(), True),
    StructField("stock", StringType(), True),
    StructField("fecha_inicio", StringType(), True),
    StructField("fecha_fin", StringType(), True)
])

spark_df_final = spark.createDataFrame(datos_totales, schema=esquema)

# 3. Mostrar resultados
spark_df_final.show(10)
print(f"Total de registros procesados: {spark_df_final.count()}")

+----+------+-----+------------+----------+
| año|precio|stock|fecha_inicio| fecha_fin|
+----+------+-----+------------+----------+
|2023| 57.13|   KO|  2023-01-01|2023-12-31|
|2023|  57.1|   KO|  2023-01-01|2023-12-31|
|2023| 56.45|   KO|  2023-01-01|2023-12-31|
|2023| 57.53|   KO|  2023-01-01|2023-12-31|
|2023| 56.82|   KO|  2023-01-01|2023-12-31|
|2023| 56.38|   KO|  2023-01-01|2023-12-31|
|2023| 56.27|   KO|  2023-01-01|2023-12-31|
|2023| 55.55|   KO|  2023-01-01|2023-12-31|
|2023| 55.75|   KO|  2023-01-01|2023-12-31|
|2023| 55.97|   KO|  2023-01-01|2023-12-31|
+----+------+-----+------------+----------+
only showing top 10 rows
Total de registros procesados: 750


In [13]:
import os

# Save the DataFrame to a Parquet file
# We use mode='overwrite' to ensure we can run this multiple times
file_path = "stock_data.parquet"
spark_df_final.write.mode("overwrite").parquet(file_path)

print(f"DataFrame saved successfully to: {os.path.abspath(file_path)}")
# List files in the parquet directory to verify
!ls -lh stock_data.parquet

DataFrame saved successfully to: /content/stock_data.parquet
total 8.0K
-rw-r--r-- 1 root root 3.5K May  7 22:00 part-00000-2a7e3611-e461-4d4f-9787-39938f3ebf9f-c000.snappy.parquet
-rw-r--r-- 1 root root 3.6K May  7 22:00 part-00001-2a7e3611-e461-4d4f-9787-39938f3ebf9f-c000.snappy.parquet
-rw-r--r-- 1 root root    0 May  7 22:00 _SUCCESS


CARGAR DATOS EN BASE DE DATOS

In [14]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 12.4 MB/s eta 0:00:00


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Re-initialize Spark Session with the PostgreSQL JDBC driver
spark = SparkSession.builder \
    .appName("StockETL_Postgres") \
    .config("spark.jars", "/content/postgresql-42.2.5.jar") \
    .getOrCreate()

print("Spark session created with PostgreSQL driver and types imported.")

Spark session created with PostgreSQL driver and types imported.


In [6]:
parquet_file_path = "stock_data.parquet"

In [7]:
try:
    # Read the data from Parquet
    parquet_df = spark.read.parquet(parquet_file_path)
    print(f"DataFrame loaded from Parquet file at '{parquet_file_path}' successfully.")
    parquet_df.show(5)

    # Updated JDBC URL (Ensure this is the public endpoint provided by Render)
    jdbc_url = "jdbc:postgresql://dpg-d20rilje5dus7385ebv0-a.oregon-postgres.render.com/dbjobs_wwxk"

    # Table name
    table_name = "stock_data_from_parquet"

    # Attempting write with SSL mode required for many cloud databases
    parquet_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", table_name) \
    .option("user", "admjobs") \
    .option("password", "lhR6LfElCVI3ti3FPt86Wumcb1idV8Ml") \
    .option("driver", "org.postgresql.Driver") \
    .option("ssl", "true") \
    .option("sslmode", "require") \
    .mode("overwrite") \
    .save()

    print(f"DataFrame saved to table '{table_name}' successfully.")

except Exception as e:
    print(f"Error connecting to the database: {e}")
    print("\nTroubleshooting Tip: Check if your Render database allows connections from external IP addresses (0.0.0.0/0).")

DataFrame loaded from Parquet file at 'stock_data.parquet' successfully.
+----+------+-----+------------+----------+
| año|precio|stock|fecha_inicio| fecha_fin|
+----+------+-----+------------+----------+
|2023| 42.28| NVDA|  2023-01-01|2023-12-31|
|2023| 42.07| NVDA|  2023-01-01|2023-12-31|
|2023| 42.47| NVDA|  2023-01-01|2023-12-31|
|2023| 42.15| NVDA|  2023-01-01|2023-12-31|
|2023| 42.37| NVDA|  2023-01-01|2023-12-31|
+----+------+-----+------------+----------+
only showing top 5 rows
Error connecting to the database: An error occurred while calling o63.save.
: org.postgresql.util.PSQLException: The connection attempt failed.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:292)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:49)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:195)
	at org.postgresql.Driver.makeConnection(Driver.java:454)
	at org.postgresql.Driver.connect(Driver.java:256)
	at